# Topic 2: Kafka + Spark Integration

> **Phase 6 → Week 11 → Topic 2**

---

## Kafka Architecture Recap

```
Producer → Topic → Consumer Group

Topic "orders":
  Partition 0: [msg0, msg3, msg6, ...]   → Spark executor 1
  Partition 1: [msg1, msg4, msg7, ...]   → Spark executor 2
  Partition 2: [msg2, msg5, msg8, ...]   → Spark executor 3

Each message:
  key:       bytes (optional, used for partitioning)
  value:     bytes (the actual payload — usually JSON, Avro, Protobuf)
  timestamp: when the message was produced
  offset:    monotonically increasing ID within a partition
  partition: which partition it belongs to
  topic:     topic name

Offset = Kafka's equivalent of a watermark for Spark to track progress
```

---

## Spark ↔ Kafka Reading Schema

```python
kafka_df = spark.readStream.format("kafka").load()

# Spark presents Kafka messages as:
kafka_df.printSchema()
# root
#  |-- key:            binary
#  |-- value:          binary    ← your payload is here
#  |-- topic:          string
#  |-- partition:      integer
#  |-- offset:         long
#  |-- timestamp:      timestamp
#  |-- timestampType:  integer

# You must decode key and value:
kafka_df.select(
    F.col("key").cast("string"),
    F.col("value").cast("string")   # then parse as JSON
)
```

---

## Interview Cheat Sheet

**Q: How does Spark read from Kafka?**
> Spark uses the Kafka consumer API internally. Each Spark executor reads from one or more Kafka partitions in parallel. The number of Spark tasks per micro-batch equals the number of Kafka partitions being consumed. Spark tracks offsets in the checkpoint — it stores the latest offset consumed per partition. On restart, it resumes from the checkpointed offsets (not from the beginning).

**Q: What is `startingOffsets` and when does it matter?**
> `startingOffsets` controls where Spark starts reading from when there is no checkpoint (first run). Options: `"earliest"` (read all history), `"latest"` (read only new messages from this moment). On subsequent runs with a checkpoint, this option is ignored — Spark always resumes from checkpointed offsets. Use `"latest"` in production to avoid reprocessing historical data on first start.

**Q: What is `kafka.max.poll.records` and why does it matter?**
> It limits how many records Spark fetches from Kafka in one poll per partition per task. Without a limit, a Kafka partition with a large backlog could return millions of records in one micro-batch, causing OOM or very long batch times. Set it to control micro-batch size and memory usage.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week11 - Kafka Integration") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)
print()
print("NOTE: This notebook shows Kafka patterns.")
print("To run live Kafka demos, start Kafka with:")
print("  docker run -p 9092:9092 apache/kafka:latest")
print("Or add a Kafka service to docker-compose.yml (see Part 5)")

## Part 1: Reading from Kafka — Pattern Reference

In [ ]:
print("""
Reading from Kafka:
════════════════════════════════════════════════════════════════

# Basic read — subscribe to a single topic
df = spark.readStream \\
    .format("kafka") \\
    .option("kafka.bootstrap.servers", "kafka-broker:9092") \\
    .option("subscribe", "orders") \\
    .option("startingOffsets", "latest") \\
    .load()

# Subscribe to multiple topics
.option("subscribe", "orders,payments,refunds")

# Subscribe via regex (all topics matching pattern)
.option("subscribePattern", "orders.*")

# Consume from specific offsets (advanced)
.option("startingOffsets",
        '{"orders":{"0":12345,"1":67890}}'  # partition:offset
)

# Limit records per partition per batch (prevent OOM on large backlogs)
.option("maxOffsetsPerTrigger", 100000)

# Security (SSL + SASL)
.option("kafka.security.protocol", "SASL_SSL")
.option("kafka.sasl.mechanism", "PLAIN")
.option("kafka.sasl.jaas.config", "...")

════════════════════════════════════════════════════════════════
""")

## Part 2: Parsing Kafka JSON Payloads

In [ ]:
# Simulate Kafka messages as binary JSON (same shape as real Kafka)
# In production: replace with spark.readStream.format("kafka")...

import json
from datetime import datetime

# Simulate the Kafka schema: key (binary), value (binary JSON)
fake_kafka = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load() \
    .withColumn("key", F.concat(F.lit("order_"), F.col("value").cast("string")).cast("binary")) \
    .withColumn("value",
        F.to_json(F.struct(
            F.concat(F.lit("O"), F.lpad(F.col("value").cast("string"), 6, "0")).alias("order_id"),
            F.concat(F.lit("C"), (F.col("value") % 10 + 1).cast("string")).alias("customer_id"),
            ((F.col("value") % 500 + 10).cast("double")).alias("amount"),
            F.element_at(
                F.array(F.lit("electronics"), F.lit("clothing"), F.lit("food")),
                (F.col("value") % 3 + 1).cast("int")
            ).alias("category"),
            F.lit("pending").alias("status"),
        ))
    ).cast("binary").alias("value")

# Define the schema of the JSON payload
order_schema = StructType([
    StructField("order_id",    StringType()),
    StructField("customer_id", StringType()),
    StructField("amount",      DoubleType()),
    StructField("category",    StringType()),
    StructField("status",      StringType()),
])

# Parse the Kafka value (binary → string → JSON → struct)
parsed = fake_kafka \
    .withColumn("key_str",   F.col("key").cast("string")) \
    .withColumn("value_str", F.col("value").cast("string")) \
    .withColumn("data",      F.from_json(F.col("value_str"), order_schema)) \
    .select(
        "timestamp",          # Kafka message timestamp
        "key_str",
        F.col("data.order_id").alias("order_id"),
        F.col("data.customer_id").alias("customer_id"),
        F.col("data.amount").alias("amount"),
        F.col("data.category").alias("category"),
        F.col("data.status").alias("status"),
    )

import time
q = parsed.writeStream \
    .format("memory") \
    .queryName("kafka_orders") \
    .outputMode("append") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(8)
print("Parsed Kafka messages:")
spark.sql("SELECT order_id, customer_id, amount, category, status FROM kafka_orders LIMIT 10").show()
q.stop()

## Part 3: Writing to Kafka

In [ ]:
print("""
Writing to Kafka:
════════════════════════════════════════════════════════════════

Kafka sink requires exactly two columns: key (optional) and value (binary).

# Step 1: serialize your data to JSON string
output = processed_df.select(
    F.col("order_id").cast("string").alias("key"),   # optional partition key
    F.to_json(F.struct("*")).alias("value")           # serialize all columns to JSON
)

# Step 2: write to Kafka topic
query = output.writeStream \\
    .format("kafka") \\
    .option("kafka.bootstrap.servers", "kafka-broker:9092") \\
    .option("topic", "processed_orders") \\
    .option("checkpointLocation", "/checkpoints/kafka_writer") \\
    .trigger(processingTime="10 seconds") \\
    .start()

Key points:
  - key determines which Kafka partition the message goes to (hash of key)
  - value MUST be string or binary
  - Use to_json(struct(*)) to serialize all columns
  - Kafka sink is AT-LEAST-ONCE — duplicates possible on retry
  - For exactly-once: use idempotent producer + transactional Kafka (requires Kafka 0.11+)

# To write to different topics dynamically (Kafka 2.6+):
output.withColumn("topic", F.when(F.col("category") == "electronics",
                                   "electronics-orders")
                            .otherwise("other-orders")) \\
       .writeStream.format("kafka")...
════════════════════════════════════════════════════════════════
""")

## Part 4: End-to-End Kafka Pipeline (Simulated)

In [ ]:
# Simulated end-to-end: rate → parse → enrich → write to Delta (simulating Kafka output)
import os, shutil

DELTA_OUT = "/tmp/kafka_pipeline_delta"
CKPT = "/tmp/ckpt_kafka_pipeline"
for p in [DELTA_OUT, CKPT]:
    if os.path.exists(p): shutil.rmtree(p)

# Simulate Kafka messages (JSON payloads)
raw_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("raw_json",
        F.to_json(F.struct(
            F.concat(F.lit("O"), F.col("value").cast("string")).alias("order_id"),
            ((F.col("value") % 999 + 1).cast("double")).alias("amount"),
            F.element_at(
                F.array(F.lit("electronics"), F.lit("clothing"), F.lit("food")),
                (F.col("value") % 3 + 1).cast("int")
            ).alias("category"),
        ))
    )

payload_schema = StructType([
    StructField("order_id",  StringType()),
    StructField("amount",    DoubleType()),
    StructField("category",  StringType()),
])

# Static enrichment table
tax_rates = spark.createDataFrame([
    ("electronics", 0.18),
    ("clothing",    0.12),
    ("food",        0.05),
], ["category", "tax_rate"])

# Parse → validate → enrich
processed = raw_stream \
    .withColumn("data", F.from_json(F.col("raw_json"), payload_schema)) \
    .select("timestamp", "data.*") \
    .filter(F.col("amount") > 0) \
    .join(tax_rates, "category", "left") \
    .withColumn("tax",         F.round(F.col("amount") * F.col("tax_rate"), 2)) \
    .withColumn("total",       F.round(F.col("amount") + F.col("tax"), 2)) \
    .withColumn("processed_at", F.current_timestamp())

batch_count = []

def write_to_delta(batch_df, batch_id):
    count = batch_df.count()
    batch_count.append(count)
    batch_df.write.format("parquet").mode("append").save(DELTA_OUT)
    print(f"  Batch {batch_id}: wrote {count} rows")

q = processed.writeStream \
    .foreachBatch(write_to_delta) \
    .option("checkpointLocation", CKPT) \
    .trigger(processingTime="3 seconds") \
    .start()

time.sleep(12)
q.stop()

total = spark.read.parquet(DELTA_OUT).count()
print(f"\nTotal rows written: {total}")
print("Sample:")
spark.read.parquet(DELTA_OUT).select("order_id", "category", "amount", "tax", "total").show(5)

## Part 5: Adding Kafka to docker-compose

In [ ]:
print("""
To run Kafka locally, add to docker-compose.yml:
════════════════════════════════════════════════════════════════

  zookeeper:
    image: confluentinc/cp-zookeeper:7.5.0
    environment:
      ZOOKEEPER_CLIENT_PORT: 2181

  kafka:
    image: confluentinc/cp-kafka:7.5.0
    depends_on: [zookeeper]
    ports:
      - "9092:9092"
    environment:
      KAFKA_ZOOKEEPER_CONNECT: zookeeper:2181
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:29092,PLAINTEXT_HOST://localhost:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,PLAINTEXT_HOST:PLAINTEXT
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1

Then add kafka-python package and spark-kafka connector:

  # In Dockerfile or requirements.txt:
  kafka-python==2.0.2

  # Spark submit with kafka connector:
  spark-submit \\
    --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 \\
    my_kafka_job.py

  # Or in SparkSession builder:
  SparkSession.builder \\
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")

Test with kafka-python:
  from kafka import KafkaProducer, KafkaConsumer
  producer = KafkaProducer(bootstrap_servers='localhost:9092')
  producer.send('orders', key=b'O001', value=b'{"amount": 100.0}')
════════════════════════════════════════════════════════════════
""")

## Exercises

1. Write the complete code to read from a Kafka topic `"payments"`, parse each value as JSON with schema `{payment_id: string, order_id: string, amount: double, currency: string}`, convert USD to INR (rate=84.0), and write the enriched output to a Parquet file sink.
2. What columns does Spark present when reading from Kafka? Why are `key` and `value` binary rather than string?
3. You have a Kafka topic with 6 partitions. How many Spark tasks run in parallel per micro-batch when reading this topic? What happens if you add 2 more partitions to the Kafka topic while the job is running?
4. Your job uses `startingOffsets=earliest` on the first run. The topic has 10M historical messages. What happens? What option limits the backlog processing rate?
5. Explain the difference between `subscribe` (single topic), `subscribe` (comma-separated), and `subscribePattern` (regex). When would you use each?